In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/insurance_clean.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [2]:
df.shape

(1337, 7)

In [3]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [4]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [5]:
print("Features:")
print(X.columns.tolist())
print("\nTarget:")
print(y.name)

Features:
['age', 'sex', 'bmi', 'children', 'smoker', 'region']

Target:
charges


In [6]:
numerical_features = [
   "age",
   "bmi",
   "children"
]
categorical_features = [
   "sex",
   "smoker",
   "region"
]
print("Numerical:", numerical_features)
print("Categorical:", categorical_features)

Numerical: ['age', 'bmi', 'children']
Categorical: ['sex', 'smoker', 'region']


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
   X,
   y,
   test_size=0.2,
   random_state=42
)
print("Training records:", len(X_train));
print("Testing records:", len(X_test))

Training records: 1069
Testing records: 268


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
numerical_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]
preprocessor = ColumnTransformer(
   transformers=[
       ("num", "passthrough", numerical_features),
       ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
   ]
)
print("Preprocessor created successfully")

Preprocessor created successfully


In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

Training data shape: (1069, 11)
Testing data shape: (268, 11)


In [10]:
feature_names = preprocessor.get_feature_names_out()

print("Processed Features:")

print(feature_names)

Processed Features:
['num__age' 'num__bmi' 'num__children' 'cat__sex_female' 'cat__sex_male'
 'cat__smoker_no' 'cat__smoker_yes' 'cat__region_northeast'
 'cat__region_northwest' 'cat__region_southeast' 'cat__region_southwest']


In [11]:
print("Target variable:", y.name)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Target variable: charges
Training target shape: (1069,)
Testing target shape: (268,)


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
print("ML models imported successfully")

ML models imported successfully


In [13]:
linear_model = LinearRegression()
linear_model.fit(
   X_train_processed,
   y_train
)
print("Linear Regression trained successfully")

Linear Regression trained successfully


In [14]:
rf_model = RandomForestRegressor(
   n_estimators=300,
   max_depth=12,
   random_state=42
)
rf_model.fit(
   X_train_processed,
   y_train
)
print("Random Forest trained successfully")

Random Forest trained successfully


In [15]:
xgb_model = XGBRegressor(
   n_estimators=300,
   learning_rate=0.05,
   max_depth=5,
   subsample=0.8,
   colsample_bytree=0.8,
   random_state=42,
   objective="reg:squarederror"
)
xgb_model.fit(
   X_train_processed,
   y_train
)
print("XGBoost trained successfully")

XGBoost trained successfully


In [16]:
linear_pred = linear_model.predict(X_test_processed)
rf_pred = rf_model.predict(X_test_processed)
xgb_pred = xgb_model.predict(X_test_processed)
print("Predictions generated successfully")

Predictions generated successfully


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
def evaluate_model(name, y_true, y_pred):
   mae = mean_absolute_error(y_true, y_pred)
   rmse = np.sqrt(mean_squared_error(y_true, y_pred))
   r2 = r2_score(y_true, y_pred)
   return {
       "Model": name,
       "MAE": mae,
       "RMSE": rmse,
       "R2 Score": r2
   }
results = []
results.append(
   evaluate_model("Linear Regression", y_test, linear_pred)
)
results.append(
   evaluate_model("Random Forest", y_test, rf_pred)
)
results.append(
   evaluate_model("XGBoost", y_test, xgb_pred)
)
results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,4177.045561,5956.342894,0.806929
1,Random Forest,2547.240310,4609.537106,0.884369
2,XGBoost,2705.955796,4598.677038,0.884914


In [18]:
best_model_name = results_df.loc[
   results_df["R2 Score"].idxmax(),
   "Model"
]
print("Best Model:", best_model_name)

Best Model: XGBoost


In [19]:
from sklearn.pipeline import Pipeline
import joblib
final_pipeline = Pipeline(
   steps=[
       ("preprocessor", preprocessor),
       ("model", xgb_model)
   ]
)
final_pipeline.fit(X_train, y_train)
print("Final insurance cost pipeline trained successfully")

Final insurance cost pipeline trained successfully


In [20]:
joblib.dump(
   final_pipeline,
   "../models/insurance_cost_model.pkl"
)
print("Model saved successfully")

Model saved successfully


In [21]:
sample_customer = pd.DataFrame({
   "age": [35],
   "sex": ["female"],
   "bmi": [28.5],
   "children": [1],
   "smoker": ["no"],
   "region": ["southeast"]
})
sample_prediction = final_pipeline.predict(sample_customer)
print("Predicted Medical Cost:", round(sample_prediction[0], 2))

Predicted Medical Cost: 4970.92
